### 벡터 DB
- 문서가 많을 때 대량 검색을 알아서 해주는 벡터 DB 구축
- 인덱싱(색인), 100만개 정도 되더라도 검색하고 계산이 거의 바로 된다
- 메타데이터(카테고리, 날짜 등)를 바탕으로 검색이 가능

Question(질문) -> Retriever(검색기) -> 벡터DB -> Generate(응답 생성) 

- 근사 검색(ANN)
- HNSW
    - 벡터들을 여러 층의 그래프로 연결해두고, 위에서 방향 잡아서 아래로 내려가며 데이터 훑는 방식

#### 벡터  DB 종류
- chromaDB
- FAISS(옛날 로컬 벡터 DB)
- Qdrant
- Pinecone(유료)
- pgvector - 기존의 postgreSQL에 플러그인만 추가하면 사용 가능
- Neo4j(GraphDB 용인데 벡터 DB에도 쓰임)
- milvus - 데이터 엄청 많을 때

In [46]:
import chromadb

print(chromadb.__version__)

1.5.9


In [47]:
import pandas as pd

df = pd.read_csv("../data/11-1_뉴스정제.csv").head(200).reset_index(drop=True)
df.head(3)

,제목,본문,카테고리,요약,출처URL,정제본문
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,경제,"6 6일 현대백화점그룹이 광주시에 문화복합몰을 만든다고 6일 밝혔으며, 광주시는 서...",https://n.news.naver.com/mnews/article/001/001...,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,전주 뉴시스 김얼 기자 이스타항공 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직...,경제,이이스항공은 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직 전 의원이 출소한 것...,https://n.news.naver.com/mnews/article/003/001...,전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 ‘10주년 기념주...,경제,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 소셜미디어 인스타...,https://n.news.naver.com/mnews/article/366/000...,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...


In [48]:
docs = df['정제본문'].tolist()

print(len(docs))

200


In [49]:
# 메타데이터로 쓸만한 것 확인 - 나중엔 필요 메타데이터는 LLM으로 처리
df[['제목', '카테고리', '출처URL']].head(3)

,제목,카테고리,출처URL
0,현대백화점그룹 더현대 광주 추진,경제,https://n.news.naver.com/mnews/article/001/001...
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,경제,https://n.news.naver.com/mnews/article/003/001...
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,경제,https://n.news.naver.com/mnews/article/366/000...


### ChromaDB 컬렉션 만들기
컬렉션 : 문서를 담는 단위 -> 이름 적어주면 됨
1. 임베딩 함수가 뭔지
2. 거리 척도가 뭔지 (우리는 코사인 유사도)

In [50]:
import os
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction  # chromaDB 자체에서 Openai 임베딩 모델 있음
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [51]:
embeddings = OpenAIEmbeddingFunction(
    model_name="text-embedding-3-small"
)

embed_client = chromadb.Client()

# collection은 
collection = embed_client.get_or_create_collection(
    name="news_collection",
    embedding_function=embeddings,
    configuration={'hnsw' : {"space" : "cosine"}}

)

In [52]:
collection

Collection(name=news_collection)

In [53]:
ids = df.index.astype(str).tolist()
metadata = df[['제목', '카테고리', '출처URL']].to_dict('records')
metadata

[{'제목': '현대백화점그룹 더현대 광주 추진',
  '카테고리': '경제',
  '출처URL': 'https://n.news.naver.com/mnews/article/001/0013293519?sid=101'},
 {'제목': '이스타항공 이상직 회사와 무관…오해 살 언동 말아야',
  '카테고리': '경제',
  '출처URL': 'https://n.news.naver.com/mnews/article/003/0011282043?sid=101'},
 {'제목': '농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트',
  '카테고리': '경제',
  '출처URL': 'https://n.news.naver.com/mnews/article/366/0000824966?sid=101'},
 {'제목': '오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은',
  '카테고리': '경제',
  '출처URL': 'https://n.news.naver.com/mnews/article/437/0000304023?sid=101'},
 {'제목': '푸르덴셜생명 더 큰 드림 변액연금보험Ⅱ에 신규펀드 13종 추가',
  '카테고리': '경제',
  '출처URL': 'https://n.news.naver.com/mnews/article/014/0004860779?sid=101'},
 {'제목': '오늘부터 사고 나면 중대재해처벌법 기소 가능성 더 커져',
  '카테고리': '경제',
  '출처URL': 'https://n.news.naver.com/mnews/article/277/0005111594?sid=101'},
 {'제목': '역세권에 신규분양 봇물…천안 부성지구 한라비발디 654가구 분양',
  '카테고리': '경제',
  '출처URL': 'https://n.news.naver.com/mnews/article/374/0000293010?sid=101'},
 {'제목': '식량 대란 오나',
  '카테고리': '경제',
  '출처URL': 'https://n

In [ ]:
# 컬렉션에 추가
collection.upsert(ids=ids, documents=docs, metadatas=metadata)  # id 잘 안넣으면 랜덤으로 하기 때문에 잘 지정해줘야?
collection.count()

200

In [55]:
# 검색기
question = "대출 정보에 대해서 알려줘"

result = collection.query(query_texts=[question], n_results=5)  # 아까 1_RAG에서우리가 구현했던 기능이 한줄로 구현됨
result

{'ids': [['111', '51', '120', '130', '3']],
 'embeddings': None,
 'documents': [['보건복지부 제공 보건복지부는 저소득 청년에 저축액의 최대 3배까지 추가 적립해주는 청년내일저축계좌 가입자를 오는 18일부터 모집한다고 밝혔다 해당 제도는 월 10만원을 저축하면 정부가 지원금 월 10만원을 추가 적립하는 방식으로 3년간 지원해 청년의 자산 형성을 도와주는 청년특별대책제도다 참여자는 3년 만기 시 본인 납입액 360만원을 포함해 총 720만원과 예금이자를 수령하게 된다 청년내일저축계좌는 신청 시점 기준 만 19 34세인 청년이 대상이다 본인 소득 가구 소득 가구 재산 등 3가지 기준을 충족해야 한다 신청 당시에 근로 중이어야 하며 근로 사업소득이 50만원 초과 200만원 이하여야 한다 또한 자신이 속한 가구의 소득이 기준 중위소득 100 2022년 4인 가구 기준 512만1080원 이하여야 한다 가구 재산 기준은 지역에 따라 차등이 있다 대도시 3억5000만원 중소도시 2억원 농어촌 1억7000만원 이하인 가구가 대상이다 신청자가 기초생활수급자이거나 차상위계층 기준 중위소득 50 이하 이라면 혜택이 훨씬 크다 참여자가 10만원을 적립할 때마다 정부 지원금 30만원이 지급된다 3년 뒤 만기 때 총 1440만원의 적립금과 예금이자를 수령하게 된다 또한 가입 가능 연령이 만 15 39세로 더 넓으며 근로 사업소득기준도 면제된다 복지부는 그동안 청년층 대상 자산형성지원 사업을 해왔으나 그 대상이 수급자와 차상위계층에만 한정됐었다 그러나 이번에는 중위소득 100 이하인 청년으로 대상이 확대되면서 신청 대상이 지난해 1만8000명에서 올해 10만4000명으로 6배 증가했다 가입을 희망하는 청년은 복지로 홈페이지 www bokjiro go kr 를 통해 신청하면 된다 정부는 원활한 신청을 위해 신청 시작일인 18일부터 2주 7월 18 29일 간은 출생일 기준으로 5부제를 시행한다 5부제 기간에 신청하지 못한 경우 

In [56]:
result['documents'][0]

['보건복지부 제공 보건복지부는 저소득 청년에 저축액의 최대 3배까지 추가 적립해주는 청년내일저축계좌 가입자를 오는 18일부터 모집한다고 밝혔다 해당 제도는 월 10만원을 저축하면 정부가 지원금 월 10만원을 추가 적립하는 방식으로 3년간 지원해 청년의 자산 형성을 도와주는 청년특별대책제도다 참여자는 3년 만기 시 본인 납입액 360만원을 포함해 총 720만원과 예금이자를 수령하게 된다 청년내일저축계좌는 신청 시점 기준 만 19 34세인 청년이 대상이다 본인 소득 가구 소득 가구 재산 등 3가지 기준을 충족해야 한다 신청 당시에 근로 중이어야 하며 근로 사업소득이 50만원 초과 200만원 이하여야 한다 또한 자신이 속한 가구의 소득이 기준 중위소득 100 2022년 4인 가구 기준 512만1080원 이하여야 한다 가구 재산 기준은 지역에 따라 차등이 있다 대도시 3억5000만원 중소도시 2억원 농어촌 1억7000만원 이하인 가구가 대상이다 신청자가 기초생활수급자이거나 차상위계층 기준 중위소득 50 이하 이라면 혜택이 훨씬 크다 참여자가 10만원을 적립할 때마다 정부 지원금 30만원이 지급된다 3년 뒤 만기 때 총 1440만원의 적립금과 예금이자를 수령하게 된다 또한 가입 가능 연령이 만 15 39세로 더 넓으며 근로 사업소득기준도 면제된다 복지부는 그동안 청년층 대상 자산형성지원 사업을 해왔으나 그 대상이 수급자와 차상위계층에만 한정됐었다 그러나 이번에는 중위소득 100 이하인 청년으로 대상이 확대되면서 신청 대상이 지난해 1만8000명에서 올해 10만4000명으로 6배 증가했다 가입을 희망하는 청년은 복지로 홈페이지 www bokjiro go kr 를 통해 신청하면 된다 정부는 원활한 신청을 위해 신청 시작일인 18일부터 2주 7월 18 29일 간은 출생일 기준으로 5부제를 시행한다 5부제 기간에 신청하지 못한 경우 8월 1 5일에 생일과 무관하게 신청할 수 있다 대상자 선정 결과는 소득 재산 조사 등을 거쳐 10월 중에 발표된다 곽숙영 복지부 복지정책관은 

In [57]:
context = "\n\n".join(result['documents'][0])   # 리스트 내용들을 \n\n과 함께 합쳐주는 기능! join 유용하게 잘 쓰자
print(context)

보건복지부 제공 보건복지부는 저소득 청년에 저축액의 최대 3배까지 추가 적립해주는 청년내일저축계좌 가입자를 오는 18일부터 모집한다고 밝혔다 해당 제도는 월 10만원을 저축하면 정부가 지원금 월 10만원을 추가 적립하는 방식으로 3년간 지원해 청년의 자산 형성을 도와주는 청년특별대책제도다 참여자는 3년 만기 시 본인 납입액 360만원을 포함해 총 720만원과 예금이자를 수령하게 된다 청년내일저축계좌는 신청 시점 기준 만 19 34세인 청년이 대상이다 본인 소득 가구 소득 가구 재산 등 3가지 기준을 충족해야 한다 신청 당시에 근로 중이어야 하며 근로 사업소득이 50만원 초과 200만원 이하여야 한다 또한 자신이 속한 가구의 소득이 기준 중위소득 100 2022년 4인 가구 기준 512만1080원 이하여야 한다 가구 재산 기준은 지역에 따라 차등이 있다 대도시 3억5000만원 중소도시 2억원 농어촌 1억7000만원 이하인 가구가 대상이다 신청자가 기초생활수급자이거나 차상위계층 기준 중위소득 50 이하 이라면 혜택이 훨씬 크다 참여자가 10만원을 적립할 때마다 정부 지원금 30만원이 지급된다 3년 뒤 만기 때 총 1440만원의 적립금과 예금이자를 수령하게 된다 또한 가입 가능 연령이 만 15 39세로 더 넓으며 근로 사업소득기준도 면제된다 복지부는 그동안 청년층 대상 자산형성지원 사업을 해왔으나 그 대상이 수급자와 차상위계층에만 한정됐었다 그러나 이번에는 중위소득 100 이하인 청년으로 대상이 확대되면서 신청 대상이 지난해 1만8000명에서 올해 10만4000명으로 6배 증가했다 가입을 희망하는 청년은 복지로 홈페이지 www bokjiro go kr 를 통해 신청하면 된다 정부는 원활한 신청을 위해 신청 시작일인 18일부터 2주 7월 18 29일 간은 출생일 기준으로 5부제를 시행한다 5부제 기간에 신청하지 못한 경우 8월 1 5일에 생일과 무관하게 신청할 수 있다 대상자 선정 결과는 소득 재산 조사 등을 거쳐 10월 중에 발표된다 곽숙영 복지부 복지정책관은 이번

#### ChromaDB 저장하기

In [58]:
# 저장소 생성
persistent_client = chromadb.PersistentClient(path="./news_chroma_db")

In [59]:
# 컬렉션 만들기
saved_collection = persistent_client.get_or_create_collection(
    name="news_collection",
    embedding_function=embeddings,
    configuration={"hnsw" : {"space" : "cosine"}}
)

In [60]:
# 마지막 적재
saved_collection.upsert(ids=ids,
                        documents=docs,
                        metadatas=metadata)

#### ChromaDB 불러오기
- 이전에 저장한 DB 불러와서 저장하기

In [ ]:
load_client = chromadb.PersistentClient(path="news_chroma_db")
load_collection = load_client.get_collection(
    name="news_collection",
    embedding_function=embeddings   # configuration 어차피 안바뀌니 적을 필요 X, 임베딩도 또 하는건 아니지만 뭘 썼는지 알려주기 위해
)


In [65]:
load_collection.count()

200

In [ ]:
load_collection.query(query_texts=["가계 대출 소식에 대해 알려줘"], n_results=5)

{'ids': [['51', '127', '188', '108', '89']],
 'embeddings': None,
 'documents': [['기사내용 요약 가계대출 잔액 699조6521억원 금리상승 자산시장 주춤 영향 하반기 규제 강화에 증가 전환 어려울 듯 서울 뉴시스 이주혜 기자 국내 주요 은행의 가계대출 잔액이 올해 상반기 감소한 것으로 나타났다 6개월간 9조원 이상이 줄었다 특히 신용대출이 8조원 넘게 줄면서 가계대출 감소세를 이끈 것으로 분석됐다 1일 은행권에 따르면 KB국민 신한 하나 우리 NH농협은행의 지난달 말 기준 가계대출 잔액은 699조6521억원으로 전월보다 1조4094억원 감소했다 가계대출 잔액이 700조원대 아래로 떨어진 것은 지난해 8월 이후 약 10개월 만이다 가계대출은 올해 1월부터 6개월 연속 감소세를 지속하면서 반년간 9조4008억원이 줄었다 은행권 관계자는 상반기 대출 감소는 금리 영향이 제일 크다고 볼 수 있다 며 자산시장도 상승세를 멈추면서 대출을 받아서 투자하는 수요가 사라졌다 실수요자가 아니면 대출을 받지 않는 분위기 라고 설명했다 주택담보대출은 지난달 소폭 늘면서 한 달 만에 증가했다 지난달 말 기준 주담대 잔액은 506조7714억원로 전월보다 991억원 늘었다 올해 들어 지난달까지 6개월간 주담대 잔액은 1조3668억원 늘어난 것으로 나타났다 지난해 하반기에는 한 달에만 3조원 이상 4조원까지도 늘어난 것과 비교하면 증가세가 둔화한 것이다 전월에는 감소세를 보이기도 했다 신용대출은 지난해 12월부터 지난달까지 꾸준히 줄면서 가계대출 감소세를 견인했다 올해에만 8조8783억원이 줄었다 지난달 말 신용대출 잔액은 130조6789억원으로 전월보다 1조1204억원이 감소했다 감소폭은 전월 6613억원 보다 커졌다 지난해까지만 해도 낮은 금리와 자산시장의 높은 수익률에 대출을 통한 영끌 로 투자에 나선 이들이 많았지만 금리가 가파르게 오르고 자산시장이 침체하면서 신용대출도 감소세를 지속하고 있다 집단대출은 지난달 말 

### Qdrant
- 방식은 거의 chromadb와 동일

In [62]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

In [63]:
qdrant = QdrantClient(path="./qdrant_news_db")   # 메모리에 하려면 ":memory:"
qdrant

RuntimeError: Storage folder ./qdrant_news_db is already accessed by another instance of Qdrant client. If you require concurrent access, use Qdrant server instead.

In [ ]:
qdrant.create_collection(
    collection_name="news_qdrant",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
)

True

### 문서 적재
- 문서를 포인트 단위로 넣는 방식
- id, vector, playload(부가정보 = 메타데이터 + 원문)

In [ ]:
client = OpenAI()

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=docs
)

In [ ]:
response.data[0].embedding

[-0.020660400390625,
 -0.0028591156005859375,
 0.034210205078125,
 0.034271240234375,
 0.01126861572265625,
 0.0037384033203125,
 0.0038433074951171875,
 0.0570068359375,
 -0.0007305145263671875,
 -0.059722900390625,
 -0.034515380859375,
 -0.0190277099609375,
 0.0014085769653320312,
 -0.04046630859375,
 0.0245513916015625,
 -0.011505126953125,
 -0.004791259765625,
 -0.0009984970092773438,
 0.0478515625,
 -0.012451171875,
 -0.0214691162109375,
 -0.02880859375,
 0.01332855224609375,
 0.006488800048828125,
 -0.007358551025390625,
 -0.018646240234375,
 0.01395416259765625,
 0.0479736328125,
 -0.0010051727294921875,
 -0.01537322998046875,
 -0.03497314453125,
 -0.03692626953125,
 0.0318603515625,
 0.015472412109375,
 0.0149078369140625,
 0.0260772705078125,
 0.0093994140625,
 0.0213623046875,
 0.0309295654296875,
 -0.00678253173828125,
 0.03057861328125,
 -0.006877899169921875,
 0.003173828125,
 0.004497528076171875,
 0.0196990966796875,
 0.0259857177734375,
 -0.0028095245361328125,
 0.00177

In [ ]:
points = []

for i in range(len(docs)):
    points.append(PointStruct(
        id=i,
        vector=response.data[i].embedding,
        payload={'제목' : df['제목'][i],
                 '카테고리' : df['카테고리'][i]}

    ))

In [ ]:
points[:5]

[PointStruct(id=0, vector=[-0.020660400390625, -0.0028591156005859375, 0.034210205078125, 0.034271240234375, 0.01126861572265625, 0.0037384033203125, 0.0038433074951171875, 0.0570068359375, -0.0007305145263671875, -0.059722900390625, -0.034515380859375, -0.0190277099609375, 0.0014085769653320312, -0.04046630859375, 0.0245513916015625, -0.011505126953125, -0.004791259765625, -0.0009984970092773438, 0.0478515625, -0.012451171875, -0.0214691162109375, -0.02880859375, 0.01332855224609375, 0.006488800048828125, -0.007358551025390625, -0.018646240234375, 0.01395416259765625, 0.0479736328125, -0.0010051727294921875, -0.01537322998046875, -0.03497314453125, -0.03692626953125, 0.0318603515625, 0.015472412109375, 0.0149078369140625, 0.0260772705078125, 0.0093994140625, 0.0213623046875, 0.0309295654296875, -0.00678253173828125, 0.03057861328125, -0.006877899169921875, 0.003173828125, 0.004497528076171875, 0.0196990966796875, 0.0259857177734375, -0.0028095245361328125, 0.001773834228515625, 0.0532

In [ ]:
qdrant.upsert(collection_name="news_qdrant",
              points=points)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:
import numpy as np
# DB 저장됐으니 이제 질문 날려보기!

def embed(texts):
    """텍스트 목록을 받아서 벡터 배열로 변환하는 함수. openai embedding을 사용해 문서 수 x 1536차원으로 변환"""

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    return np.array([item.embedding for item in response.data])

In [ ]:
question = "대출 정보에 대해서 알려줘"

query_vec = embed([question])[0]

result = qdrant.query_points(
    collection_name="news_qdrant",   # 아까 만든 collection과 이름 같아야
    query=query_vec.tolist(),
    limit=5
)

print(result)

points=[ScoredPoint(id=111, version=0, score=0.3627825727441777, payload={'제목': '월 10만원 저축하면 정부가 10만원 더…18일부터 청년내일저축계좌 모집', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=51, version=0, score=0.3620797837926796, payload={'제목': '은행 가계대출 반년간 9조 넘게 줄었다', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=120, version=0, score=0.3417419887667473, payload={'제목': '문과생도 IT 인재로 육성…취업률 59% ‘눈길’', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=130, version=0, score=0.3183405259790597, payload={'제목': '뉴스프라임 처음 집 사면 LTV 80%…하반기 달라지는 정책은', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3, version=0, score=0.30419494012366255, payload={'제목': '오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None)]


In [ ]:
for point in result.points:
    print(point.id)
    print(docs[point.id][:100]) # 해당 인덱스 가진 본문
    print(point.payload)    # 메타데이터들

111
보건복지부 제공 보건복지부는 저소득 청년에 저축액의 최대 3배까지 추가 적립해주는 청년내일저축계좌 가입자를 오는 18일부터 모집한다고 밝혔다 해당 제도는 월 10만원을 저축하면 정
{'제목': '월 10만원 저축하면 정부가 10만원 더…18일부터 청년내일저축계좌 모집', '카테고리': '경제'}
51
기사내용 요약 가계대출 잔액 699조6521억원 금리상승 자산시장 주춤 영향 하반기 규제 강화에 증가 전환 어려울 듯 서울 뉴시스 이주혜 기자 국내 주요 은행의 가계대출 잔액이 올
{'제목': '은행 가계대출 반년간 9조 넘게 줄었다', '카테고리': '경제'}
120
KBS 부산 앵커 문과를 나와 죄송합니다 라는 자조적 말이 유행할 정도로 문과생의 취업 문턱은 갈수록 높아지는 상황인데요 기업이 직접 가르쳐서 뽑는 식으로 비전공자들도 IT 회사 
{'제목': '문과생도 IT 인재로 육성…취업률 59% ‘눈길’', '카테고리': '경제'}
130
방송 2022년 7월 1일 금 이슈오늘 진행 박진형 이나연 출연 윤석천 경제평론가 오늘부터 유류세율 인하폭이 확대됐고 생애 최초 구매자를 대상으로 한 주택담보대출비율도 완화됐습니다
{'제목': '뉴스프라임 처음 집 사면 LTV 80%…하반기 달라지는 정책은', '카테고리': '경제'}
3
img tag s 지난 30일 서울의 한 주유소 사진 연합뉴스 img tag e 오늘 1일 부터 유류세율 인하 폭이 확대됩니다 올해 3분기에는 생애최초 주택 구매 대출 규제도 완화
{'제목': '오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은', '카테고리': '경제'}


### 실습
1. 구축하고 싶은 문서를 찾기
2. chromadb에 문서 적재하고
3. 질문에 답하는 RAG 시스템을 구축해보자!

In [ ]:
# 1. 데이터 가져오기
data_library = pd.read_csv("../data/전국도서관표준데이터.csv", encoding="cp949")
data_library = data_library[:200]
docs = data_library['휴관일'].tolist()

In [ ]:
# 2 chromadb 컬렉션 만들기
collection = embed_client.get_or_create_collection(
    name="library_collection",
    embedding_function=embeddings,
    configuration={'hnsw' : {"space" : "cosine"}}
)

In [ ]:
# 정보 뽑기
ids = data_library.index.astype(str).tolist()
metadata = data_library[['도서관명', '시도명', '시군구명']].to_dict('records')

# 컬렉션에 추가
collection.upsert(ids=ids, documents=docs, metadatas=metadata)  # id 잘 안넣으면 랜덤으로 하기 때문에 잘 지정해줘야?
collection.count()

200

In [ ]:
question = "경남 진해의 "

result = collection.query(query_texts=[question], n_results=5)  # 아까 1_RAG에서 우리가 구현했던 기능이 한줄로 구현됨
result